In [1]:
import requests

url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.cond": "cancer",
    "pageSize": 100,
    "format": "json"
}

r = requests.get(url, params=params)
trials = r.json()["studies"]

print(f"Got {len(trials)} trials")
print(trials[0]["protocolSection"]["eligibilityModule"]["eligibilityCriteria"][:500])

Got 100 trials
Inclusion Criteria:

* Patient from 1 to 55 years old (Children and adolescents from 1 to 17 years/ Adults from 18 to 55 years)
* Patients with 1st ALL relapse, which could be either isolated bone marrow relapse, or combined (medullary and extra-medullary) relapse, or extra-medullary isolated relapse; or lymphoblastic lymphoma (excepted Burkitt lymphoma) OR Failure to ALL first line treatment (no complete remission obtained)
* Patient previously treated with free E.Coli L-asparaginase form or pe


In [2]:
import json

with open("data/trials_raw.json", "w") as f:
    json.dump(trials, f, indent=2)

print("Data saved!")
print(f"Total trials saved: {len(trials)}")

Data saved!
Total trials saved: 100


In [3]:
import pandas as pd

records = []

for trial in trials:
    try:
        nct_id = trial["protocolSection"]["identificationModule"]["nctId"]
        title = trial["protocolSection"]["identificationModule"]["briefTitle"]
        eligibility = trial["protocolSection"]["eligibilityModule"]["eligibilityCriteria"]
        records.append({
            "nct_id": nct_id,
            "title": title,
            "eligibility_text": eligibility
        })
    except KeyError:
        continue

df = pd.DataFrame(records)
print(f"Extracted {len(df)} trials")
print(df.head(3))

Extracted 100 trials
        nct_id                                              title  \
0  NCT01518517  GRASPA (Erythrocytes Encapsulating L-asparagin...   
1  NCT06855589  Optimal Duration of Hormonal Therapy for Unfav...   
2  NCT05263050  Trial of an Alternative Cabozantinib Dosing Sc...   

                                    eligibility_text  
0  Inclusion Criteria:\n\n* Patient from 1 to 55 ...  
1  Inclusion Criteria:\n\n* 1\. Histologically co...  
2  Inclusion Criteria\n\n* Cohorts A and B: Histo...  


In [4]:
import pandas as pd

records = []

for trial in trials:
    try:
        nct_id = trial["protocolSection"]["identificationModule"]["nctId"]
        title = trial["protocolSection"]["identificationModule"]["briefTitle"]
        eligibility = trial["protocolSection"]["eligibilityModule"]["eligibilityCriteria"]
        records.append({
            "nct_id": nct_id,
            "title": title,
            "eligibility_text": eligibility
        })
    except KeyError:
        continue

df = pd.DataFrame(records)
print(f"Extracted {len(df)} trials")
print(df.head(3))

Extracted 100 trials
        nct_id                                              title  \
0  NCT01518517  GRASPA (Erythrocytes Encapsulating L-asparagin...   
1  NCT06855589  Optimal Duration of Hormonal Therapy for Unfav...   
2  NCT05263050  Trial of an Alternative Cabozantinib Dosing Sc...   

                                    eligibility_text  
0  Inclusion Criteria:\n\n* Patient from 1 to 55 ...  
1  Inclusion Criteria:\n\n* 1\. Histologically co...  
2  Inclusion Criteria\n\n* Cohorts A and B: Histo...  


In [5]:
df.to_csv("data/trials_clean.csv", index=False)
print("Clean data saved to data/trials_clean.csv!")

Clean data saved to data/trials_clean.csv!


In [7]:
# How long is the eligibility text typically?
df["text_length"] = df["eligibility_text"].str.len()

print("=== Data Summary ===")
print(f"Total trials: {len(df)}")
print(f"Average text length: {df['text_length'].mean():.0f} characters")
print(f"Shortest text: {df['text_length'].min()} characters")
print(f"Longest text: {df['text_length'].max()} characters")
print("\n=== Sample eligibility text ===")
print(df['eligibility_text'][0][:800])

=== Data Summary ===
Total trials: 100
Average text length: 2182 characters
Shortest text: 108 characters
Longest text: 9518 characters

=== Sample eligibility text ===
Inclusion Criteria:

* Patient from 1 to 55 years old (Children and adolescents from 1 to 17 years/ Adults from 18 to 55 years)
* Patients with 1st ALL relapse, which could be either isolated bone marrow relapse, or combined (medullary and extra-medullary) relapse, or extra-medullary isolated relapse; or lymphoblastic lymphoma (excepted Burkitt lymphoma) OR Failure to ALL first line treatment (no complete remission obtained)
* Patient previously treated with free E.Coli L-asparaginase form or pegylated one
* Performance Status ≤ 2 (WHO score)
* Patient informed and consent provided (the 2 parents need to consent when children are below 18)

Exclusion Criteria:

* ALL t(9;22) and/or BCR-ABL positive (Philadelphia chromosome positive)
* Patient with 2nd relapse and over
* Women of childbeari


In [8]:
import re

labeled_criteria = []

for _, row in df.iterrows():
    text = row["eligibility_text"]
    nct_id = row["nct_id"]
    
    # Split into inclusion and exclusion sections
    # Try to find where exclusion starts
    split_patterns = [
        r"(?i)exclusion criteria",
        r"(?i)key exclusion",
        r"(?i)exclude"
    ]
    
    exclusion_start = len(text)  # default: no exclusion found
    for pattern in split_patterns:
        match = re.search(pattern, text)
        if match:
            exclusion_start = match.start()
            break
    
    inclusion_text = text[:exclusion_start]
    exclusion_text = text[exclusion_start:]
    
    # Split into individual bullet points
    def extract_bullets(block):
        lines = re.split(r'\n[\*\-\d•]+\.?\s+', block)
        return [l.strip() for l in lines if len(l.strip()) > 30]
    
    for criterion in extract_bullets(inclusion_text):
        labeled_criteria.append({
            "nct_id": nct_id,
            "criterion": criterion,
            "label": "inclusion"
        })
    
    for criterion in extract_bullets(exclusion_text):
        labeled_criteria.append({
            "nct_id": nct_id,
            "criterion": criterion,
            "label": "exclusion"
        })

labeled_df = pd.DataFrame(labeled_criteria)
print(f"Total labeled criteria: {len(labeled_df)}")
print(f"\nLabel distribution:")
print(labeled_df["label"].value_counts())
print(f"\nSample inclusion:")
print(labeled_df[labeled_df["label"]=="inclusion"]["criterion"].iloc[0])
print(f"\nSample exclusion:")
print(labeled_df[labeled_df["label"]=="exclusion"]["criterion"].iloc[0])

Total labeled criteria: 1340

Label distribution:
label
exclusion    736
inclusion    604
Name: count, dtype: int64

Sample inclusion:
Patient from 1 to 55 years old (Children and adolescents from 1 to 17 years/ Adults from 18 to 55 years)

Sample exclusion:
ALL t(9;22) and/or BCR-ABL positive (Philadelphia chromosome positive)


In [9]:
# Save labeled data
labeled_df.to_csv("data/labeled_criteria.csv", index=False)
print("Labeled data saved!")
print(f"\nFinal dataset shape: {labeled_df.shape}")

Labeled data saved!

Final dataset shape: (1340, 3)
